# AgriNexus AI — Research-Grade Notebook 01: Crop Recommendation System

**Task**: Multi-Class Crop Selection (22 Classes) based on Soil Nutrient Levels ($N, P, K$) and Micro-Climatic Parameters
**Primary Dataset**: `Crop_recommendation.csv` (2,200 Observations, 7 Features)
**Scientific Focus**: SHA-256 Duplicate Hashing Audit, Stratified Partitioning, Multi-Model Suite Benchmarking (Dummy Baseline, Naive Bayes, Linear Models, Trees, Boosting), Calibration (Expected Calibration Error ECE), Top-3/Top-5 Accuracy, Isolation-Forest-Based Distributional Anomaly Detection & Training-Distribution Plausibility Warning, Perturbation Robustness ($\%\text{ Label Stability}$), and Model Artifact Serialization/Reload Verification.

In [1]:
# Section 1: Environment, Dependencies & Deterministic Seed Setup
import os
import sys
import time
import random
import hashlib
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, IsolationForest
import xgboost as xgb
import lightgbm as lgb

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, log_loss, confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')

# Deterministic Seed Setup
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

DATA_PATH = Path('d:/PROJECTS/AGRINEXUS-AI/data/raw/crop_recommendation/Crop_recommendation.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('../data/raw/crop_recommendation/Crop_recommendation.csv')

MODELS_DIR = Path('d:/PROJECTS/AGRINEXUS-AI/Notebook/models')
if not MODELS_DIR.exists():
    MODELS_DIR = Path('models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Environment Ready | Seed: {SEED}")
print(f"Data Path: {DATA_PATH.resolve()}")
print(f"Models Directory: {MODELS_DIR.resolve()}")

Environment Ready | Seed: 42
Data Path: D:\PROJECTS\AGRINEXUS-AI\data\raw\crop_recommendation\Crop_recommendation.csv
Models Directory: D:\PROJECTS\AGRINEXUS-AI\Notebook\models


## 2. Problem Statement & Agronomic Feature Range Audit
Multi-class crop recommendation aims to classify the most suitable crop out of 22 target categories given soil nutrient inputs ($N, P, K$) and local climatic metrics (Temperature, Humidity, pH, Rainfall).

### Expected Agronomic Ranges:
- Nitrogen ($N$): 0 to 140 kg/ha
- Phosphorus ($P$): 5 to 145 kg/ha
- Potassium ($K$): 5 to 205 kg/ha
- Temperature: 8.0 to 45.0 °C
- Humidity: 14.0 to 100.0 %
- Soil pH: 3.5 to 10.0
- Rainfall: 20.0 to 300.0 mm

In [2]:
# Section 3: Data Ingestion, Integrity & SHA-256 Duplicate Audit
df_raw = pd.read_csv(DATA_PATH)
print(f"Raw Dataset Loaded: {len(df_raw):,} rows, {len(df_raw.columns)} columns")

# Missing values check
assert df_raw.isnull().sum().sum() == 0, "Missing values detected in dataset!"
print("Missing Value Audit Passed: 0 missing values.")

# SHA-256 Exact Duplicate Audit
def compute_row_hash(row):
    row_str = "_".join([str(val) for val in row.values])
    return hashlib.sha256(row_str.encode('utf-8')).hexdigest()

row_hashes = df_raw.apply(compute_row_hash, axis=1)
exact_duplicates = row_hashes.duplicated()
num_duplicates = exact_duplicates.sum()
print(f"SHA-256 Exact Duplicates Detected: {num_duplicates}")

df_clean = df_raw[~exact_duplicates].copy()
feature_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
target_col = 'label'

for col in feature_cols:
    print(f"  - Feature {col:<12}: min={df_clean[col].min():7.2f}, max={df_clean[col].max():7.2f}")

class_counts = df_clean[target_col].value_counts()
print(f"Total Crop Classes: {len(class_counts)} | Samples per class: {class_counts.min()} to {class_counts.max()}")

Raw Dataset Loaded: 2,200 rows, 8 columns
Missing Value Audit Passed: 0 missing values.
SHA-256 Exact Duplicates Detected: 0
  - Feature N           : min=   0.00, max= 140.00
  - Feature P           : min=   5.00, max= 145.00
  - Feature K           : min=   5.00, max= 205.00
  - Feature temperature : min=   8.83, max=  43.68
  - Feature humidity    : min=  14.26, max=  99.98
  - Feature ph          : min=   3.50, max=   9.94
  - Feature rainfall    : min=  20.21, max= 298.56
Total Crop Classes: 22 | Samples per class: 100 to 100


In [3]:
# Section 4: Stratified Train / Validation / Test Split & Preprocessing Setup
X = df_clean[feature_cols].copy()
y_str = df_clean[target_col].copy()

le = LabelEncoder()
y = le.fit_transform(y_str)
class_names = list(le.classes_)

X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"Partition Sizes | Train: {len(X_train_raw)} (70%) | Val: {len(X_val_raw)} (15%) | Test: {len(X_test_raw)} (15%)")

# Scaler fitted strictly on training partition
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled = scaler.transform(X_val_raw)
X_test_scaled = scaler.transform(X_test_raw)

# Isolation Forest fitted strictly on training features for anomaly scoring
iso_forest = IsolationForest(contamination=0.01, random_state=SEED)
iso_forest.fit(X_train_raw)

print("Scaler and Isolation Forest fitted strictly on training partition.")

Partition Sizes | Train: 1540 (70%) | Val: 330 (15%) | Test: 330 (15%)
Scaler and Isolation Forest fitted strictly on training partition.


In [4]:
# Section 5: Candidate Classifier Suite Benchmarking & Calibration Audit
def compute_ece(y_true, y_prob, n_bins=10):
    confidences = np.max(y_prob, axis=1)
    predictions = np.argmax(y_prob, axis=1)
    accuracies = predictions == y_true
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        bin_lower, bin_upper = bin_boundaries[i], bin_boundaries[i+1]
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = np.mean(in_bin)
        if prop_in_bin > 0:
            accuracy_in_bin = np.mean(accuracies[in_bin])
            avg_confidence_in_bin = np.mean(confidences[in_bin])
            ece += np.abs(accuracy_in_bin - avg_confidence_in_bin) * prop_in_bin
    return ece

def compute_top_k_acc(y_true, y_prob, k=3):
    top_k_preds = np.argsort(y_prob, axis=1)[:, -k:]
    hits = [y_true[i] in top_k_preds[i] for i in range(len(y_true))]
    return np.mean(hits)

candidate_models = {
    "Dummy Baseline (Stratified)": DummyClassifier(strategy="stratified", random_state=SEED),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Gaussian Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, random_state=SEED, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, max_depth=12, random_state=SEED, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=100, random_state=SEED),
    "XGBoost Classifier": xgb.XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=6, random_state=SEED, n_jobs=-1),
    "LightGBM Classifier": lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=6, random_state=SEED, n_jobs=-1, verbose=-1)
}

benchmark_results = []
best_val_f1 = -1.0
best_model_name = None
best_model = None

print("Evaluating Candidate Models on Validation Partition...")
for name, clf in candidate_models.items():
    if "Dummy" in name or "Logistic" in name or "Naive" in name:
        clf.fit(X_train_scaled, y_train)
        y_val_pred = clf.predict(X_val_scaled)
        y_val_prob = clf.predict_proba(X_val_scaled)
    else:
        clf.fit(X_train_raw, y_train)
        y_val_pred = clf.predict(X_val_raw)
        y_val_prob = clf.predict_proba(X_val_raw)
        
    val_acc = accuracy_score(y_val, y_val_pred)
    prec, rec, val_f1, _ = precision_recall_fscore_support(y_val, y_val_pred, average='macro', zero_division=0)
    val_ece = compute_ece(y_val, y_val_prob)
    val_top3 = compute_top_k_acc(y_val, y_val_prob, k=3)
    
    benchmark_results.append({
        "Model": name,
        "Val Acc": val_acc,
        "Val Macro Precision": prec,
        "Val Macro Recall": rec,
        "Val Macro F1": val_f1,
        "Val Top-3 Acc": val_top3,
        "Val ECE": val_ece
    })
    print(f"  {name:<28} | Val Acc: {val_acc*100:6.2f}% | Val Macro F1: {val_f1:7.4f} | ECE: {val_ece:6.4f}")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_name = name
        best_model = clf

df_bench = pd.DataFrame(benchmark_results)
print(f"\nCHAMPION MODEL SELECTED (via Validation Macro F1): {best_model_name} (Val Macro F1 = {best_val_f1:.4f})")

Evaluating Candidate Models on Validation Partition...
  Dummy Baseline (Stratified)  | Val Acc:   3.94% | Val Macro F1:  0.0398 | ECE: 0.9606
  Logistic Regression          | Val Acc:  96.67% | Val Macro F1:  0.9665 | ECE: 0.1363
  Gaussian Naive Bayes         | Val Acc:  99.39% | Val Macro F1:  0.9939 | ECE: 0.0065
  Random Forest                | Val Acc:  99.09% | Val Macro F1:  0.9909 | ECE: 0.0517
  Extra Trees                  | Val Acc:  99.70% | Val Macro F1:  0.9970 | ECE: 0.1754
  HistGradientBoosting         | Val Acc:  98.79% | Val Macro F1:  0.9879 | ECE: 0.0103
  XGBoost Classifier           | Val Acc:  99.39% | Val Macro F1:  0.9939 | ECE: 0.0331
  LightGBM Classifier          | Val Acc:  98.79% | Val Macro F1:  0.9878 | ECE: 0.0100

CHAMPION MODEL SELECTED (via Validation Macro F1): Extra Trees (Val Macro F1 = 0.9970)


In [5]:
# Section 6: Held-Out Unseen Test Set Evaluation & Plausibility Audit
if "Dummy" in best_model_name or "Logistic" in best_model_name or "Naive" in best_model_name:
    y_test_pred = best_model.predict(X_test_scaled)
    y_test_prob = best_model.predict_proba(X_test_scaled)
else:
    y_test_pred = best_model.predict(X_test_raw)
    y_test_prob = best_model.predict_proba(X_test_raw)

test_acc = accuracy_score(y_test, y_test_pred)
test_prec, test_rec, test_macro_f1, _ = precision_recall_fscore_support(y_test, y_test_pred, average='macro', zero_division=0)
_, _, test_weighted_f1, _ = precision_recall_fscore_support(y_test, y_test_pred, average='weighted', zero_division=0)
test_ece = compute_ece(y_test, y_test_prob)
test_top3 = compute_top_k_acc(y_test, y_test_prob, k=3)
test_top5 = compute_top_k_acc(y_test, y_test_prob, k=5)
test_loss = log_loss(y_test, y_test_prob)

print(f"Held-Out Test Results for Champion ({best_model_name}):")
print(f"  - Test Accuracy:        {test_acc * 100:.2f}%")
print(f"  - Test Macro Precision: {test_prec:.4f}")
print(f"  - Test Macro Recall:    {test_rec:.4f}")
print(f"  - Test Macro F1:        {test_macro_f1:.4f}")
print(f"  - Test Weighted F1:     {test_weighted_f1:.4f}")
print(f"  - Test Top-3 Accuracy:  {test_top3 * 100:.2f}%")
print(f"  - Test Top-5 Accuracy:  {test_top5 * 100:.2f}%")
print(f"  - Test Log Loss:        {test_loss:.4f}")
print(f"  - Calibration ECE:      {test_ece:.4f}")

# Training-Distribution Plausibility Warning Audit (Z-Score > 4.0)
train_mean = X_train_raw.mean(axis=0)
train_std = X_train_raw.std(axis=0) + 1e-6
z_scores = np.abs((X_test_raw - train_mean) / train_std)
extreme_z_outliers = (z_scores > 4.0).any(axis=1)

# Isolation-Forest-Based Distributional Anomaly Detection
iso_preds = iso_forest.predict(X_test_raw)
iso_anomalies = (iso_preds == -1)

print(f"\nTraining-Distribution Plausibility & Anomaly Audit:")
print(f"  - Z-Score > 4.0 Training-Distribution Plausibility Warnings: {extreme_z_outliers.sum()} / {len(X_test_raw)}")
print(f"  - Isolation-Forest-Based Distributional Anomalies Flagged:   {iso_anomalies.sum()} / {len(X_test_raw)}")

# Feature Noise Perturbation Robustness Test
np.random.seed(SEED)
noise_scale = 0.05  # 5% relative Gaussian perturbation
X_test_perturbed = X_test_raw * (1.0 + np.random.normal(0, noise_scale, size=X_test_raw.shape))
if "Dummy" in best_model_name or "Logistic" in best_model_name or "Naive" in best_model_name:
    y_pert_pred = best_model.predict(scaler.transform(X_test_perturbed))
else:
    y_pert_pred = best_model.predict(X_test_perturbed)

label_retention_rate = np.mean(y_test_pred == y_pert_pred)
print(f"  - Feature Noise Perturbation Label Stability (5% noise): {label_retention_rate * 100:.2f}%")

Held-Out Test Results for Champion (Extra Trees):
  - Test Accuracy:        98.79%
  - Test Macro Precision: 0.9890
  - Test Macro Recall:    0.9879
  - Test Macro F1:        0.9878
  - Test Weighted F1:     0.9878
  - Test Top-3 Accuracy:  100.00%
  - Test Top-5 Accuracy:  100.00%
  - Test Log Loss:        0.2141
  - Calibration ECE:      0.1668

Training-Distribution Plausibility & Anomaly Audit:
  - Z-Score > 4.0 Training-Distribution Plausibility Warnings: 0 / 330
  - Isolation-Forest-Based Distributional Anomalies Flagged:   5 / 330
  - Feature Noise Perturbation Label Stability (5% noise): 98.18%


In [6]:
# Section 7: Model Artifact Serialization & Reload Verification
artifact_filename = "crop_recommendation.pkl"
artifact_path = MODELS_DIR / artifact_filename

pipeline_dict = {
    'scaler': scaler,
    'model': best_model,
    'label_encoder': le,
    'best_model_name': best_model_name,
    'feature_cols': feature_cols,
    'class_names': class_names,
    'train_mean': train_mean.to_dict(),
    'train_std': train_std.to_dict(),
    'isolation_forest': iso_forest,
    'metadata': {
        'dataset_name': 'Crop Recommendation Dataset',
        'train_samples': len(X_train_raw),
        'val_samples': len(X_val_raw),
        'test_samples': len(X_test_raw),
        'test_accuracy': float(test_acc),
        'test_macro_f1': float(test_macro_f1),
        'test_top3_accuracy': float(test_top3),
        'test_ece': float(test_ece),
        'perturbation_label_retention': float(label_retention_rate),
        'random_seed': SEED,
        'saved_at': time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())
    }
}

with open(artifact_path, 'wb') as f:
    pickle.dump(pipeline_dict, f)

artifact_size_mb = artifact_path.stat().st_size / (1024 * 1024)
print("Artifact Overwritten Successfully!")
print(f"  - Path: {artifact_path.resolve()}")
print(f"  - Size: {artifact_size_mb:.2f} MB")

# Reload Verification Check
with open(artifact_path, 'rb') as f:
    reloaded_dict = pickle.load(f)

reloaded_scaler = reloaded_dict['scaler']
reloaded_model = reloaded_dict['model']

X_sample = X_test_raw.iloc[:10]
if "Dummy" in best_model_name or "Logistic" in best_model_name or "Naive" in best_model_name:
    y_orig_sample = best_model.predict(scaler.transform(X_sample))
    y_reload_sample = reloaded_model.predict(reloaded_scaler.transform(X_sample))
else:
    y_orig_sample = best_model.predict(X_sample)
    y_reload_sample = reloaded_model.predict(X_sample)

is_deterministic = np.array_equal(y_orig_sample, y_reload_sample)
print(f"\nArtifact Reload Verification Check: Predictions Match 100%: {is_deterministic}")
assert is_deterministic, "CRITICAL FAILURE: Reloaded artifact predictions do not match!"
print("QUALITY GATE PASSED: Crop recommendation artifact reloaded cleanly.")

Artifact Overwritten Successfully!
  - Path: D:\PROJECTS\AGRINEXUS-AI\Notebook\models\crop_recommendation.pkl
  - Size: 9.27 MB

Artifact Reload Verification Check: Predictions Match 100%: True
QUALITY GATE PASSED: Crop recommendation artifact reloaded cleanly.


In [7]:
# Section 8: Final Scientific Audit Table & Conclusions
readiness = "PASS" if (test_macro_f1 >= 0.85 and is_deterministic) else "CONDITIONAL"

final_audit_summary = [
    {"Metric / Aspect": "Dataset", "Audit Value": "Crop_recommendation.csv"},
    {"Metric / Aspect": "Dataset Size", "Audit Value": f"{len(df_clean):,} total ({len(X_train_raw):,} train, {len(X_val_raw):,} val, {len(X_test_raw):,} test)"},
    {"Metric / Aspect": "Target Variable", "Audit Value": "label (Multi-class crop selection)"},
    {"Metric / Aspect": "Target Classes", "Audit Value": f"{len(class_names)} classes ({class_names[:3]}...)"},
    {"Metric / Aspect": "Features", "Audit Value": f"{len(feature_cols)} features ({', '.join(feature_cols)})"},
    {"Metric / Aspect": "Split Strategy", "Audit Value": "Stratified 70% Train / 15% Val / 15% Test Split"},
    {"Metric / Aspect": "Leakage Audit", "Audit Value": f"PASS (SHA-256 exact duplicates dropped: {num_duplicates}, Scaler fit on Train only)"},
    {"Metric / Aspect": "Baseline Model", "Audit Value": "DummyClassifier (Stratified) / Gaussian NB"},
    {"Metric / Aspect": "Candidate Models", "Audit Value": "Dummy, LogisticReg, GaussianNB, RF, ExtraTrees, HistGB, XGBoost, LightGBM"},
    {"Metric / Aspect": "Champion Model", "Audit Value": f"{best_model_name} (Selected via Validation Macro F1)"},
    {"Metric / Aspect": "Validation Metric", "Audit Value": f"Val Macro F1 = {best_val_f1:.4f}"},
    {"Metric / Aspect": "Final Test Metric", "Audit Value": f"Test Acc = {test_acc*100:.2f}%, Macro F1 = {test_macro_f1:.4f}, ECE = {test_ece:.4f}"},
    {"Metric / Aspect": "Top-K Accuracy", "Audit Value": f"Top-3 Acc = {test_top3*100:.2f}%, Top-5 Acc = {test_top5*100:.2f}%"},
    {"Metric / Aspect": "Robustness (5% Noise)", "Audit Value": f"PASS ({label_retention_rate*100:.2f}% prediction stability retention)"},
    {"Metric / Aspect": "OOD / Plausibility Audit", "Audit Value": "Isolation-Forest-based distributional anomaly detection & Plausibility Warning"},
    {"Metric / Aspect": "Artifact Reload Result", "Audit Value": "PASS (Exact deterministic output match)"},
    {"Metric / Aspect": "Known Limitations", "Audit Value": "Ideal experimental crop distributions require out-of-distribution density checks in real field deployment"},
    {"Metric / Aspect": "Readiness Status", "Audit Value": readiness}
]

df_audit_summary = pd.DataFrame(final_audit_summary)
print("="*70)
print("FINAL MODEL AUDIT REPORT — CROP RECOMMENDATION")
print("="*70)
print(df_audit_summary.to_string(index=False))
print("="*70)

FINAL MODEL AUDIT REPORT — CROP RECOMMENDATION
         Metric / Aspect                                                                                               Audit Value
                 Dataset                                                                                   Crop_recommendation.csv
            Dataset Size                                                              2,200 total (1,540 train, 330 val, 330 test)
         Target Variable                                                                        label (Multi-class crop selection)
          Target Classes                                                          22 classes (['apple', 'banana', 'blackgram']...)
                Features                                                 7 features (N, P, K, temperature, humidity, ph, rainfall)
          Split Strategy                                                           Stratified 70% Train / 15% Val / 15% Test Split
           Leakage Audit            